In [10]:
from pyspark.sql.functions import *
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Explode").getOrCreate()

data = [
    (1, ["a","b"], [{"name":"A","age":20}, {"name":"B","age":25}]),
    (2, None, None),
    (3, [], [])
]
schema = StructType([
    StructField("id" , LongType() , True),
    StructField("arr" , ArrayType(StringType()) , True),
    StructField("info" , ArrayType(StructType([
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True)
    ])) , True)
])

df = spark.createDataFrame(data, schema)
df.printSchema()
df.show(truncate=False)

root
 |-- id: long (nullable = true)
 |-- arr: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- info: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- age: integer (nullable = true)

+---+------+------------------+
|id |arr   |info              |
+---+------+------------------+
|1  |[a, b]|[{A, 20}, {B, 25}]|
|2  |NULL  |NULL              |
|3  |[]    |[]                |
+---+------+------------------+



In [20]:
df.select(
    "id",

    # explode
    # explode("info").alias("explode_val"),

    # explode_outer
    # explode_outer("arr").alias("explode_outer_val"),

    # # posexplode
    # posexplode("arr").alias("pos1","posexplode_val"),

    # # posexplode_outer
    # posexplode_outer("arr").alias("pos2","posexplode_outer_val"),

    # # inline
    inline("info"),

    # # inline_outer
    # inline_outer("info")
).show(truncate=False)

+---+----+---+
|id |name|age|
+---+----+---+
|1  |A   |20 |
|1  |B   |25 |
+---+----+---+



In [ ]:
# explode        → normal
# explode_outer  → keep null
# posexplode     → add position
# posexplode_outer → position + keep null
# inline         → array<struct>
# inline_outer   → array<struct> + keep null


# When you use multiple explode-type functions in same select:

# Spark creates Cartesian expansion

# Row count increases heavily.

In [25]:
# map fucntions

from pyspark.sql.functions import *

from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("marks", MapType(StringType(), IntegerType()), True)
])

data = [
    (1, {"math": 80, "eng": 75}),
    (2, {"math": 90, "eng": 85}),
    (3, {"math": 70, "eng": 60})
]

df = spark.createDataFrame(data, schema)
df.show(truncate=False)

+---+-----------------------+
|id |marks                  |
+---+-----------------------+
|1  |{math -> 80, eng -> 75}|
|2  |{math -> 90, eng -> 85}|
|3  |{math -> 70, eng -> 60}|
+---+-----------------------+



In [23]:
# all map funcitons :

df.select(
    col("id"),

    # Access by key
    col("marks")["math"].alias("math_marks"),
    # col("marks"["eng"].alias())

    # element_at
    element_at("marks", "eng").alias("eng_marks"),
    # element_at("marks" , "eng").alais()

    # Keys & Values
    map_keys("marks").alias("keys"),
    map_values("marks").alias("values"),

    # Convert map -> array(struct)
    map_entries("marks").alias("entries"),

    # Count pairs
    size("marks").alias("total_subjects"),

    # Add new key using map_concat
    map_concat(
        col("marks"),
        create_map(lit("science"), lit(100))
    ).alias("merged_map"),

    # Filter map (keep marks > 80)
    map_filter("marks", lambda k, v: v > 80).alias("filtered_map"),

    # Transform values (add 5 marks)
    transform_values("marks", lambda k, v: v + 5).alias("updated_marks"),

    # Transform keys (uppercase keys)
    transform_keys("marks", lambda k, v: upper(k)).alias("upper_keys_map")

).show(truncate=False)

+---+----------+---------+-----------+--------+-----------------------+--------------+---------------------------------------+-----------------------+-----------------------+-----------------------+
|id |math_marks|eng_marks|keys       |values  |entries                |total_subjects|merged_map                             |filtered_map           |updated_marks          |upper_keys_map         |
+---+----------+---------+-----------+--------+-----------------------+--------------+---------------------------------------+-----------------------+-----------------------+-----------------------+
|1  |80        |75       |[math, eng]|[80, 75]|[{math, 80}, {eng, 75}]|2             |{math -> 80, eng -> 75, science -> 100}|{}                     |{math -> 85, eng -> 80}|{MATH -> 80, ENG -> 75}|
|2  |90        |85       |[math, eng]|[90, 85]|[{math, 90}, {eng, 85}]|2             |{math -> 90, eng -> 85, science -> 100}|{math -> 90, eng -> 85}|{math -> 95, eng -> 90}|{MATH -> 90, ENG -> 85}|
|3  |